# 04 — Multimodal fusion + `MultimodalSCDModel`

End-to-end sanity check: synthetic cohort → `StandardPreprocessor` → batch tensors → `MultimodalSCDModel` with each fusion strategy (`attention`, `cross`, `late`).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koneke55/Mmvlm4SCD/blob/main/notebooks/04-multimodal-fusion.ipynb)

**Google Colab + GPU:** [Unsloth](https://unsloth.ai) documents a practical [Google Colab workflow](https://docs.unsloth.ai/get-started/install/google-colab) (free **T4** GPU tier, Runtime menu, run cells in order). Use it as the reference for attaching hardware acceleration.

**Note:** This repo does **not** depend on the `unsloth` pip package—only standard PyTorch + `pip install -e .`; the Unsloth guide covers Colab compute ergonomics.

**Local:** run `pip install -e .` from the repo root. **Colab:** run the environment cell below (clone under `/content` when needed).


## 1. Environment setup (Colab or local)

- **Colab:** optional `MMVLM_REPO_URL` for your fork; defaults to upstream.
- Installs this package editable (`pip install -e .`).


In [ ]:
import os
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "Could not find mmvlm4scd package root (missing src/mmvlm4scd). "
        "Open the notebook from the repo or run the Colab clone cell."
    )


if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/koneke55/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)


## 2. Accelerator check

Mirrors the GPU verification pattern recommended alongside [Unsloth's Colab instructions](https://docs.unsloth.ai/get-started/install/google-colab).


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime — for GPU follow Unsloth's Colab guide (Runtime → Change runtime type).")


## 3. Imports


In [ ]:
import torch
from torch.utils.data import DataLoader

from mmvlm4scd.data import (
    MultimodalSCDDataset,
    StandardPreprocessor,
    generate_synthetic_cohort,
)
from mmvlm4scd.data.synthetic import SCDSyntheticConfig
from mmvlm4scd.models import ModelConfig, MultimodalSCDModel


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=128, seed=0))
pre = StandardPreprocessor().fit(cohort["clinical"])
x_clin = pre.transform(cohort["clinical"])

ds = MultimodalSCDDataset(
    clinical=x_clin,
    genomic=cohort["genomic"],
    imaging=cohort["imaging"],
    temporal=cohort["temporal"],
    severity=cohort["severity"],
    survival_time=cohort["survival_time"],
    survival_event=cohort["survival_event"],
)
loader = DataLoader(ds, batch_size=32, shuffle=False, drop_last=False)
batch = next(iter(loader))
{k: v.shape for k, v in batch.items()}


In [ ]:
def build_model(fusion: str):
    cfg = ModelConfig(
        clinical_input_dim=x_clin.shape[1],
        genomic_input_dim=cohort["genomic"].shape[1],
        imaging_input_dim=cohort["imaging"].shape[1],
        temporal_input_dim=cohort["temporal"].shape[2],
        embed_dim=64,
        fusion=fusion,
    )
    return MultimodalSCDModel(cfg)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

for fusion in ("attention", "cross", "late"):
    model = build_model(fusion).to(device)
    b = {k: v.to(device) for k, v in batch.items()}
    out = model(b)
    print(fusion, "| embedding", tuple(out["embedding"].shape),
          "| severity_logits", tuple(out["severity_logits"].shape),
          "| risk_score", tuple(out["risk_score"].shape))


## Optional: one optimizer step (same stack as unit tests)

Demonstrates that gradients flow through all fusion modes.


In [ ]:
import torch.nn.functional as F

from mmvlm4scd.training.losses import cox_partial_likelihood_loss


def one_step(fusion: str):
    torch.manual_seed(0)
    model = build_model(fusion).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    b = {k: v.to(device) for k, v in batch.items()}
    opt.zero_grad(set_to_none=True)
    out = model(b)
    loss_cls = F.cross_entropy(out["severity_logits"], b["severity"])
    loss_cox = cox_partial_likelihood_loss(
        out["risk_score"], b["survival_time"], b["survival_event"]
    )
    (loss_cls + 0.1 * loss_cox).backward()
    opt.step()
    return float(loss_cls.detach()), float(loss_cox.detach())

for fusion in ("attention", "cross", "late"):
    lc, lx = one_step(fusion)
    print(f"{fusion}: CE={lc:.4f} Cox={lx:.4f}")
